In [ ]:

# Q1: Dataset Splitting Strategy
from sklearn.model_selection import train_test_split

GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

all_recipes = [
    {"genre": genre, "index": i}
    for genre in GENRES
    for i in range(100)
]
print(f"Total recipes: {len(all_recipes)}")  # 10 × 100 = 1000

train_recipes, val_recipes = train_test_split(
    all_recipes, test_size=0.2, shuffle=True, random_state=42
)

print(f"Train recipes: {len(train_recipes)}")
print(f"Val   recipes: {len(val_recipes)}")


Total recipes: 1000
Train recipes: 800
Val   recipes: 200


In [ ]:

import numpy as np

SR = 16000
DURATION = 10
N_SAMPLES = SR * DURATION  # 160,000

# Simulate loading 4 stems + 1 noise file and padding/truncating to N_SAMPLES
stems = [np.random.randn(N_SAMPLES).astype(np.float32) for _ in range(4)]
noise = np.random.randn(N_SAMPLES).astype(np.float32)

# Sum 4 stems, then add noise at weight 0.2
mix = sum(stems) + 0.2 * noise

print(f"mix.shape: {mix.shape}")   # (160000,)
print(f"mix.ndim:  {mix.ndim}")    # 1D — required by the feature extractor


mix.shape: (160000,)
mix.ndim:  1


In [ ]:

# Q3: Hugging Face Feature Extractor Output Shape
import numpy as np
from transformers import AutoFeatureExtractor

AST_MODEL = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL)

# Dummy 10-second audio at 16kHz
mix = np.ones(160000, dtype=np.float32)

inputs = feature_extractor(mix, sampling_rate=16000, return_tensors="pt")
tensor = inputs["input_values"].squeeze(0)  # remove batch dim

print(f"input_values shape after squeeze: {list(tensor.shape)}")


input_values shape after squeeze: [1024, 128]


In [ ]:

from transformers import ASTForAudioClassification

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True,  # replaces the 527-class head with 10-class head
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable parameters: 86,196,490


In [ ]:

# Q5: Inference Normalization Math
import numpy as np

y_test = np.array([-0.85, 0.40, 0.20, -0.10])

# Normalize to [-1, 1] to prevent clipping before feature extraction
y_norm = y_test / (np.max(np.abs(y_test)) + 1e-9)

print(f"Normalized array: {np.round(y_norm, 3)}")
print(f"Index 0 value:    {y_norm[0]:.3f}")


Normalized array: [-1.     0.471  0.235 -0.118]
Index 0 value:    -1.000


## Training: Fine-tune AST for Genre Classification

Full pipeline: on-the-fly stem mixing → AST feature extraction → fine-tuned classifier head → W&B logging.

In [ ]:

# ─── Imports & Config ─────────────────────────────────────────────────────────
import os, gc, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoFeatureExtractor, ASTForAudioClassification
import wandb

# Paths (works locally and on Kaggle)
if Path("/kaggle/input").exists():
    DATA_DIR   = Path("/kaggle/input/messy-mashup")
    STEMS_DIR  = DATA_DIR / "genres_stems"
    ESC50_DIR  = DATA_DIR / "ESC-50-master/audio"
    TEST_CSV   = DATA_DIR / "test.csv"
else:
    DATA_DIR   = Path("/Users/sanskar/dev/DL-GenAI-P/messy_mashup")
    STEMS_DIR  = DATA_DIR / "genres_stems"
    ESC50_DIR  = DATA_DIR / "ESC-50-master/audio"
    TEST_CSV   = DATA_DIR / "test.csv"

GENRES      = ["blues","classical","country","disco","hiphop",
               "jazz","metal","pop","reggae","rock"]
LABEL2IDX   = {g: i for i, g in enumerate(GENRES)}
IDX2LABEL   = {i: g for g, i in LABEL2IDX.items()}
AST_MODEL   = "MIT/ast-finetuned-audioset-10-10-0.4593"
SR          = 16000          # AST expects 16kHz
DURATION    = 10             # seconds per sample
N_SAMPLES   = SR * DURATION  # 160,000
SEED        = 42

WANDB_PROJECT = "23f3003478-t12026"
WANDB_ENTITY  = "23f3003478-iit-madras"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Config ready. DATA_DIR:", DATA_DIR)


In [ ]:

# ─── Build Recipe List & Split ────────────────────────────────────────────────
def build_recipes(n_per_genre=100):
    """
    Build a list of (genre, song_dir) recipe dicts without loading audio.
    Each recipe is resolved on-the-fly in the Dataset __getitem__.
    """
    recipes = []
    for genre in GENRES:
        genre_dir = STEMS_DIR / genre
        song_dirs = sorted([d for d in genre_dir.iterdir() if d.is_dir()])
        for song_dir in song_dirs[:n_per_genre]:
            recipes.append({"genre": genre, "song_dir": str(song_dir)})
    return recipes

all_recipes = build_recipes(n_per_genre=100)
train_recipes, val_recipes = train_test_split(
    all_recipes, test_size=0.2, shuffle=True, random_state=SEED
)

print(f"Total: {len(all_recipes)} | Train: {len(train_recipes)} | Val: {len(val_recipes)}")

# Pre-load ESC-50 noise paths
esc50_paths = sorted(ESC50_DIR.glob("*.wav"))[:300]
print(f"ESC-50 noise files: {len(esc50_paths)}")


In [ ]:

# ─── Dataset: On-the-Fly Stem Mixing ─────────────────────────────────────────
def load_and_pad(path, sr=SR, n_samples=N_SAMPLES):
    """Load audio at target SR, pad/truncate to exactly n_samples."""
    y, _ = librosa.load(path, sr=sr, duration=DURATION, mono=True)
    if len(y) < n_samples:
        y = np.pad(y, (0, n_samples - len(y)))
    else:
        y = y[:n_samples]
    return y.astype(np.float32)


class MashupDataset(Dataset):
    """
    On-the-fly synthetic mashup generator.
    Each sample: loads stems from one song dir, adds ESC-50 noise, returns
    the AST feature tensor and genre label.
    """

    def __init__(self, recipes, esc50_paths, feature_extractor,
                 noise_weight=0.2, augment=True):
        self.recipes          = recipes
        self.esc50_paths      = esc50_paths
        self.feature_extractor = feature_extractor
        self.noise_weight     = noise_weight
        self.augment          = augment
        self.stem_names       = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

    def __len__(self):
        return len(self.recipes)

    def __getitem__(self, idx):
        recipe   = self.recipes[idx]
        genre    = recipe["genre"]
        song_dir = Path(recipe["song_dir"])
        label    = LABEL2IDX[genre]

        # Load 4 stems and sum them into a mix
        mix = np.zeros(N_SAMPLES, dtype=np.float32)
        for stem in self.stem_names:
            stem_path = song_dir / stem
            if stem_path.exists():
                y = load_and_pad(stem_path)
                # Random volume jitter during training (±6 dB)
                if self.augment:
                    gain = 10 ** (random.uniform(-6, 6) / 20)
                    y = y * gain
                mix += y

        # Add ESC-50 noise
        noise_path = random.choice(self.esc50_paths)
        noise = load_and_pad(str(noise_path))
        mix = mix + self.noise_weight * noise

        # Normalize to prevent clipping (same formula used at inference)
        mix = mix / (np.max(np.abs(mix)) + 1e-9)

        # Extract AST input features (returns [1024, 128] tensor)
        inputs = self.feature_extractor(
            mix, sampling_rate=SR, return_tensors="pt"
        )
        input_values = inputs["input_values"].squeeze(0)  # (1024, 128)

        return input_values, label


# Quick shape sanity-check
feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL)
sample_ds = MashupDataset(train_recipes[:2], esc50_paths, feature_extractor)
x, y = sample_ds[0]
print(f"Feature tensor shape: {x.shape}  label: {y} ({IDX2LABEL[y]})")


In [ ]:

# ─── Model + DataLoaders ──────────────────────────────────────────────────────
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print(f"Device: {device}")

# Fine-tune AST with 10-class head (replaces original 527-class AudioSet head)
model = ASTForAudioClassification.from_pretrained(
    AST_MODEL,
    num_labels=10,
    ignore_mismatched_sizes=True,
)
model = model.to(device)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# DataLoaders
BATCH_SIZE  = 8
NUM_WORKERS = 2

train_ds = MashupDataset(train_recipes, esc50_paths, feature_extractor, augment=True)
val_ds   = MashupDataset(val_recipes,   esc50_paths, feature_extractor, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:

# ─── Training Loop ────────────────────────────────────────────────────────────
NUM_EPOCHS = 10
LR         = 2e-5   # low LR for fine-tuning a pretrained transformer
PATIENCE   = 4

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-7)

wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name="ast-finetuned",
    config={
        "model": "ASTForAudioClassification",
        "pretrained": AST_MODEL,
        "num_labels": 10,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "optimizer": "AdamW",
        "scheduler": "CosineAnnealingLR",
        "n_train": len(train_recipes),
        "n_val": len(val_recipes),
    },
    tags=["ast", "fine-tune", "milestone5"],
)

best_f1    = -1.0
no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(input_values=batch_x, labels=batch_y)
        loss    = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

    train_loss = total_loss / len(train_loader)
    train_f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    train_acc  = accuracy_score(all_labels, all_preds)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            outputs = model(input_values=batch_x, labels=batch_y)
            total_loss += outputs.loss.item()
            all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    val_loss = total_loss / len(val_loader)
    val_f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    val_acc  = accuracy_score(all_labels, all_preds)

    scheduler.step()
    elapsed = time.time() - t0

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
          f"Train L:{train_loss:.3f} F1:{train_f1:.4f} | "
          f"Val L:{val_loss:.3f} F1:{val_f1:.4f} Acc:{val_acc:.4f} | "
          f"{elapsed:.0f}s")

    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss, "train/f1": train_f1, "train/accuracy": train_acc,
        "val/loss": val_loss,     "val/f1": val_f1,     "val/accuracy": val_acc,
        "learning_rate": scheduler.get_last_lr()[0],
    })

    # Save best checkpoint
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "ast_finetuned_best.pt")
        no_improve = 0
        print(f"  ✓ Best model saved (val F1={best_f1:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

wandb.log({"best_val_f1": best_f1})
wandb.finish()
print(f"\nTraining complete. Best val F1: {best_f1:.4f}")


## Inference & Submission

Load best checkpoint, run on test files, generate `submission.csv`.

In [ ]:

# ─── Inference ────────────────────────────────────────────────────────────────
# Load best checkpoint
model.load_state_dict(torch.load("ast_finetuned_best.pt", map_location=device))
model.eval()

test_df = pd.read_csv(TEST_CSV)
MASHUPS_DIR = DATA_DIR / "mashups"
N_CROPS     = 3   # test-time augmentation: average over 3 evenly-spaced crops

predictions = []

with torch.no_grad():
    for _, row in test_df.iterrows():
        audio_path = MASHUPS_DIR / row["filename"]

        try:
            y_full, _ = librosa.load(str(audio_path), sr=SR, mono=True)
        except Exception as e:
            print(f"Warning: failed to load {row['filename']}: {e}")
            predictions.append({"id": row["id"], "label": GENRES[0]})
            continue

        # Normalize to prevent clipping
        y_full = y_full / (np.max(np.abs(y_full)) + 1e-9)

        # Multi-crop TTA: take N_CROPS evenly-spaced 10s windows
        crop_logits = []
        for ci in range(N_CROPS):
            if len(y_full) <= N_SAMPLES:
                chunk = np.pad(y_full, (0, N_SAMPLES - len(y_full)))
            else:
                max_start = len(y_full) - N_SAMPLES
                start     = int(ci * max_start / max(N_CROPS - 1, 1))
                chunk     = y_full[start : start + N_SAMPLES]

            inputs = feature_extractor(chunk, sampling_rate=SR, return_tensors="pt")
            input_values = inputs["input_values"].to(device)
            logits = model(input_values=input_values).logits  # (1, 10)
            crop_logits.append(logits.squeeze(0).cpu().numpy())

        # Average logits across crops → argmax
        avg_logits = np.mean(crop_logits, axis=0)
        pred_label = IDX2LABEL[int(np.argmax(avg_logits))]
        predictions.append({"id": row["id"], "label": pred_label})

submission = pd.DataFrame(predictions)
submission.to_csv("submission.csv", index=False)
print(f"submission.csv saved — {len(submission)} rows")
print(submission["label"].value_counts())
